# Mínimos Cuadrados Generalizados (MCG)

**Unidad 1.e del temario** · **Notas de Clase: capítulo 4, §4.3**

MCO supone que $\mathbb{E}[\boldsymbol\varepsilon\boldsymbol\varepsilon'] = \sigma^2 I$.
Cuando esa matriz no es escalar —heterocedasticidad, autocorrelación— MCO sigue siendo
insesgado y consistente, pero **deja de ser eficiente y sus errores estándar dejan de ser
válidos**.

MCG resuelve el problema transformando el modelo para restaurar el supuesto:

$$\hat{\boldsymbol\beta}_{MCG} = (\mathbf{X}'\Omega^{-1}\mathbf{X})^{-1}\mathbf{X}'\Omega^{-1}\mathbf{y}$$

Este cuaderno usa el caso que el propio Nerlove (1963) analizó: su función de costos
presenta heterocedasticidad ligada al tamaño de la empresa, y él mismo la corrigió
agrupando las empresas por producto.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan, het_white

import matplotlib.pyplot as plt

AZUL, NARANJA, GRIS = "#2a78d6", "#eb6834", "#8a8a8a"
plt.rcParams.update({
    "font.family": "serif", "font.size": 10,
    "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110,
})

datos = pd.read_stata("nerlove63.dta").sort_values("output").reset_index(drop=True)
for columna in ["totcost", "output", "plabor", "pfuel", "pkap"]:
    datos["l_" + columna] = np.log(datos[columna])

X = sm.add_constant(datos[["l_output", "l_plabor", "l_pfuel", "l_pkap"]])
y = datos["l_totcost"]

print(f"N = {len(datos)}   (empresas ordenadas por tamaño de producto)")

## Paso 1 — MCO y el diagnóstico

Empezamos por el modelo del cuaderno
[`Regresion_Lineal.ipynb`](Regresion_Lineal.ipynb) y le preguntamos si el supuesto de
homocedasticidad se sostiene.

In [ ]:
mco = sm.OLS(y, X).fit()

print(f"MCO:  beta_output = {mco.params['l_output']:.4f}"
      f"   (ee {mco.bse['l_output']:.4f})")
print(f"      economías de escala 1/beta = {1 / mco.params['l_output']:.4f}")

In [ ]:
bp = het_breuschpagan(mco.resid, X)
white = het_white(mco.resid, X)

print(f"Breusch-Pagan : LM = {bp[0]:7.3f}   p = {bp[1]:.6f}")
print(f"White         : LM = {white[0]:7.3f}   p = {white[1]:.6f}")
print()
print("Ambas pruebas rechazan la homocedasticidad de manera contundente.")

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.0))
ax.scatter(datos["l_output"], mco.resid, s=18, color=AZUL, alpha=0.65,
           edgecolor="none")
ax.axhline(0, color=GRIS, lw=0.9)
ax.set_xlabel("$\\ln(\\mathrm{producto})$")
ax.set_ylabel("residual de MCO")
ax.set_title("La dispersión del residual depende del tamaño de la empresa",
             loc="left", fontsize=11)
ax.text(0.5, -1.35, "las empresas pequeñas tienen residuales\nmucho más dispersos",
        fontsize=9, style="italic", color="#1a1a1a")
plt.tight_layout()
plt.show()

El patrón es inequívoco: los residuales de las empresas pequeñas están mucho más
dispersos. Tiene sentido económico —una planta pequeña tiene costos más volátiles en
términos relativos— y es justamente lo que Nerlove observó.

## Paso 2 — MCG factible con varianza por grupos

Nerlove agrupó las 145 empresas en **cinco grupos de 29**, ordenadas por producto, y
supuso varianza constante dentro de cada grupo. Es un caso de MCG **factible**: $\Omega$
no se conoce, se estima.

$$\hat\sigma_g^2 = \frac{1}{n_g}\sum_{i \in g} \hat\varepsilon_i^2,
\qquad w_i = 1/\hat\sigma_{g(i)}^2$$

In [ ]:
datos["grupo"] = np.repeat(np.arange(5), 29)

varianzas = datos.assign(e2=mco.resid ** 2).groupby("grupo")["e2"].mean()
tabla = pd.DataFrame({
    "producto_medio": datos.groupby("grupo")["output"].mean(),
    "varianza": varianzas,
})
print(tabla.round(4).to_string())
print(f"\nRazón entre la varianza mayor y la menor: {varianzas.max() / varianzas.min():.1f}")

In [ ]:
pesos = 1 / datos["grupo"].map(varianzas)
mcg = sm.WLS(y, X, weights=pesos).fit()

comparacion = pd.DataFrame({
    "MCO": mco.params, "ee_MCO": mco.bse,
    "MCG": mcg.params, "ee_MCG": mcg.bse,
})
comparacion.round(4)

## Paso 3 — Qué cambió, y qué no

Conviene distinguir tres cosas que suelen confundirse.

In [ ]:
robusto = mco.get_robustcov_results(cov_type="HC1")

print("Elasticidad costo-producto:")
print(f"  MCO           : {mco.params['l_output']:.4f}   ee = {mco.bse['l_output']:.4f}")
print(f"  MCO + robusto : {mco.params['l_output']:.4f}   ee = {robusto.bse[1]:.4f}")
print(f"  MCG factible  : {mcg.params['l_output']:.4f}   ee = {mcg.bse['l_output']:.4f}")

print("\nEconomías de escala (1/beta):")
print(f"  MCO : {1 / mco.params['l_output']:.4f}")
print(f"  MCG : {1 / mcg.params['l_output']:.4f}")

**1. Los errores estándar de MCO estaban mal.** El error robusto de White es casi el
**doble** del clásico (0.033 contra 0.018). Los intervalos de confianza que MCO reporta
por omisión son demasiado estrechos: la inferencia era inválida, aunque el punto
estimado fuera consistente.

**2. El punto estimado también se mueve.** MCG lleva la elasticidad de 0.720 a 0.800, y
con ella las economías de escala de 1.39 a 1.25. No es un ajuste cosmético: cambia la
magnitud de la conclusión económica.

**3. Robustez y eficiencia no son lo mismo, y resuelven problemas distintos:**

| | Corrige el punto estimado | Corrige la inferencia | Requiere conocer $\Omega$ |
|---|:--:|:--:|:--:|
| MCO | — | — | no |
| MCO + errores robustos | no | **sí** | no |
| MCG factible | **sí** (eficiencia) | **sí** | sí, hay que modelarla |

Los errores robustos son el remedio **seguro**: no exigen saber de dónde viene la
heterocedasticidad. MCG es más **eficiente**, pero sólo si el modelo de $\Omega$ es
correcto. Un $\Omega$ mal especificado empeora las cosas respecto de MCO con errores
robustos.

> **Por qué MCG cambia el punto estimado aquí.** MCG pondera cada observación por el
> inverso de su varianza: da más peso a las empresas grandes, cuyos residuales son más
> precisos. Si la relación fuera exactamente log-lineal, ambos estimadores convergerían al
> mismo valor. Que difieran tanto sugiere que la **forma funcional Cobb-Douglas no ajusta
> igual de bien en todo el rango de tamaños** — que es precisamente la crítica que llevó a
> Christensen y Greene (1976) a proponer la forma translogarítmica.

In [ ]:
fig, ax = plt.subplots(figsize=(7.0, 3.6))
metodos = ["MCO\n(clásico)", "MCO\n(robusto)", "MCG\nfactible"]
betas = [mco.params["l_output"], mco.params["l_output"], mcg.params["l_output"]]
errores = [mco.bse["l_output"], robusto.bse[1], mcg.bse["l_output"]]
colores = [GRIS, AZUL, NARANJA]

for i, (b, e, c) in enumerate(zip(betas, errores, colores)):
    ax.errorbar(i, b, yerr=1.96 * e, fmt="o", color=c, capsize=5, ms=7, lw=1.8)

ax.set_xticks(range(3))
ax.set_xticklabels(metodos)
ax.set_ylabel("elasticidad costo-producto")
ax.set_title("El intervalo, no sólo el punto", loc="left", fontsize=11)
ax.set_xlim(-0.5, 2.5)
plt.tight_layout()
plt.show()

La figura resume el capítulo: el punto estimado por MCO y por MCO robusto es **el mismo**
—lo único que cambia es la anchura del intervalo— mientras que MCG mueve ambos.

---

## Ejercicios

1. **Sensibilidad al número de grupos.** Repite el MCG con 3 y con 10 grupos. ¿Cuánto se
   mueve $\hat\beta$? Si se mueve mucho, ¿qué dice eso sobre la confianza que merece el
   modelo de $\Omega$?
2. **Heterocedasticidad multiplicativa.** En lugar de grupos, modela
   $\sigma_i^2 = \sigma^2 \, \text{output}_i^{\,\gamma}$ estimando $\gamma$ con una
   regresión auxiliar de $\ln \hat\varepsilon_i^2$ sobre $\ln(\text{output}_i)$. Compara
   con el MCG por grupos.
3. **MCG iterado.** Recalcula las varianzas con los residuales de MCG y reestima, hasta
   convergencia. ¿Se estabiliza? Bajo normalidad el límite es máxima verosimilitud
   (capítulo 6).
4. **La prueba de Hausman como diagnóstico.** MCO es consistente aunque $\Omega$ esté mal
   especificada; MCG sólo si está bien. ¿Puedes construir un contraste sobre esa base?
   Compáralo con el uso que se le da en `Clase_04_DatosPanel`.
5. **Lo que MCG no arregla.** El coeficiente del precio del capital sigue siendo negativo
   después de la corrección. Verifícalo. ¿Por qué la heterocedasticidad no era la causa de
   esa anomalía? *Pista:* ¿qué supuesto del modelo lineal clásico viola un signo
   equivocado, y cuál viola la heterocedasticidad?

---

## Referencias

- **Nerlove, M. (1963).** «Returns to scale in electricity supply», en *Measurement in
  Economics*, Stanford University Press.
- **Christensen, L. R. y W. H. Greene (1976).** «Economies of scale in U.S. electric power
  generation», *Journal of Political Economy* 84(4): 655-676.
- **White, H. (1980).** «A heteroskedasticity-consistent covariance matrix estimator and a
  direct test for heteroskedasticity», *Econometrica* 48(4): 817-838.

---
Parte del curso de **Econometría I**, Facultad de Ciencias, UNAM.
Teoría en el **capítulo 4, §4.3** de las Notas de Clase.